# 사전준비

# 1. 모델 불러오기

## 1) 베이스 모델 불러오기

In [13]:
# 설치 후 [런타임] > [런타임 다시 시작]을 한번 실행한 뒤 아래 셀부터 진행.
%pip install -U unsloth trl transformers accelerate bitsandbytes datasets

  Using cached trl-1.7.0-py3-none-any.whl.metadata (11 kB)
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)


In [25]:
import os, tempfile
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024           # 한 번에 처리할 최대 토큰 수(데이터 길이에 맞게 조정, 클수록 VRAM 많이 사용)
dtype = None                    # GPU에 맞게 자동 선택(None 권장)
load_in_4bit = True             # 4bit 압축 로드 여부(True= VRAM 절약, VRAM 부족 시 반드시 True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",       # 사용할 베이스 모델 이름(허깅페이스 모델 ID)
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

print(f"모델 로드 완료!")
print(f"   dtype : {next(model.parameters()).dtype}")
print(f"   device: {next(model.parameters()).device}")


==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
모델 로드 완료!
   dtype : torch.float16
   device: cuda:0


## 2) LoRA 어댑터 준비

In [26]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,                             # LoRA rank(클수록 학습 용량 많음, 보통 8,16,32,64 중 선택 )
    target_modules = [                  # 어댑터에 붙일 레이어 선택
        # Instruct 파인튜닝은 경험적으로 "q_proj", "v_proj" 두 개만으로도 충분한 효율을 냄
        # 그러나 실습에서는 전체 레이어를 포함해 학습 효과를 눈에 띄게 확인하기 위해 모두 포함.
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj",
        "down_proj"
    ],
    lora_alpha = 32,    # LoRA 학습 강도(보통 r과 동일하게 설정)
    lora_dropout = 0,   # 뉴런 드랍 비율
    bias = "none",      # 바이어스 학습 여부
    use_gradient_checkpointing = "unsloth",     # VRAM 절약 방법
    random_state = 3407,            # 시드값 고정
    use_rslora = False,
    loftq_config = None,
)

print("LoRA 어댑터 추가 완료!")


LoRA 어댑터 추가 완료!


# 2. 데이터셋 불러오기

## 1) 데이터셋 로드

In [28]:
from datasets import  load_dataset

dataset = load_dataset(
    "json",                                     # JSON/JSONL 형식의 데이터셋 로더 사용
    data_files="/content/custom_dataset.json"  # 불러올 데이터 파일 경로
)

print(f"데이터 수:{len(dataset)}")
print(f"컬럼: {dataset.column_names}")
print(dataset)

데이터 수:1
컬럼: {'train': ['messages']}
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 56
    })
})


### 2) 데이터 분할

In [29]:
split = dataset['train'].train_test_split(test_size=0.2, seed=42)
split

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 44
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 12
    })
})

In [30]:
train_dataset = split['train']
valid_dataset = split['test']

In [31]:
train_dataset[0]

{'messages': [{'role': 'system',
   'content': '당신은 온라인 쇼핑몰의 친절하고 신속한 고객 상담사입니다. 고객의 문의에 공감하며 정중하고 이해하기 쉽게 답변해주세요.'},
  {'role': 'user', 'content': '주문 후 결제 수단을 바꿀 수 있나요?'},
  {'role': 'assistant',
   'content': '결제 완료 후에는 결제 수단을 직접 변경하기 어려운 경우가 많아요 😊 보통은 기존 주문을 취소한 뒤 원하는 결제 수단으로 다시 주문해주셔야 합니다. 출고 전이라면 빠르게 처리해보시는 걸 추천드려요!'}]}

In [32]:
tokenizer.apply_chat_template(
    [
        {'role': 'system', 'content': '당신은 온라인 쇼핑몰의 친절하고 신속한 고객 상담사입니다. 고객의 문의에 공감하며 정중하고 이해하기 쉽게 답변해주세요.'},
        {'role': 'user', 'content': '주문 후 결제 수단을 바꿀 수 있나요?'},
        {'role': 'assistant','content': '결제 완료 후에는 결제 수단을 직접 변경하기 어려운 경우가 많아요 😊 보통은 기존 주문을 취소한 뒤 원하는 결제 수단으로 다시 주문해주셔야 합니다. 출고 전이라면 빠르게 처리해보시는 걸 추천드려요!'}
    ],
    tokenize=False,
    add_generation_prompt=False     # 파인튜닝 데이터 생성/확인 시
)


'<|im_start|>system\n당신은 온라인 쇼핑몰의 친절하고 신속한 고객 상담사입니다. 고객의 문의에 공감하며 정중하고 이해하기 쉽게 답변해주세요.<|im_end|>\n<|im_start|>user\n주문 후 결제 수단을 바꿀 수 있나요?<|im_end|>\n<|im_start|>assistant\n결제 완료 후에는 결제 수단을 직접 변경하기 어려운 경우가 많아요 😊 보통은 기존 주문을 취소한 뒤 원하는 결제 수단으로 다시 주문해주셔야 합니다. 출고 전이라면 빠르게 처리해보시는 걸 추천드려요!<|im_end|>\n'

## 3) formatting_func 만들기

In [33]:
# instruction/output 데이터셋을 모델이 학습할 수 있는 채팅 형식 문자열로 바꾸는 함수
def formatting_prompts_func(data):
    result = []
    messages = data['messages']

    # messages가 딕셔너리 리스트인지 확인
    if isinstance(messages[0], dict):
        # 단일 샘플로 들어온 경우
        chat = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        result.append(chat)

    else:
        # 배치로 들어온 경우
        for msgs in messages:
            chat = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False
        )
        result.append(chat)
    return result

# 3. Trainer 만들기

## 1) TrainingArgument

In [35]:
from trl import SFTConfig
from unsloth import is_bfloat16_supported

args = SFTConfig(
    per_device_train_batch_size = 2,            # ✅GPU당 배치 크기(VRAM 맞게 조정, OOM 시 줄이기)
    gradient_accumulation_steps = 4,            # ✅실질 배치 = 2 * 4 = 8(배치 크기 줄인 만큼 스텝 수 늘리기)
    warmup_steps = 5,                           # lr 워밍업 스텝수(전체 스텝의 5~10% 권장)
    # max_steps = 100,                          # ✅학습 총 스텝 수(빠른 테스트 용)
    num_train_epochs = 50,                      # ✅전체 에포크 수 (EarlyStopping 사용 시 넉넉하게 설정)
    learning_rate = 2e-4,                       # ✅학습률(LoRA 권장값 : 1e-4 ~ 3e-4)
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,                        # 과적합 방지 정규화
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "/content/ouputs",              # ✅체크포인트 저장폴터(colab 경로)
    # save_strategy = "no",                      # 중간 체크포인트 저장 활성화 할건지(colab PicklingError 방지)
    report_to = "none",                          # 로그 기록 위치
    average_tokens_across_devices = False,       # 멀티 GPU 토큰 평균화 (단일 GPU 시 False)
    load_best_model_at_end = True,               # EarlyStopping 사용 시 필수 (없으면 에러 발생)
    eval_strategy="epoch",                       # ✅ 검증 주기 (epoch마다 val loss 계산)
    save_strategy="epoch"                        # ✅ 체크포인트 저장 주기(eval_strategy와 맞춰야 함)
)


## 2) Early Stopping

In [22]:
from transformers import EarlyStoppingCallback

callbacks = [
    EarlyStoppingCallback(
        early_stopping_patience = 3         # 3번 연속 loss값이 안 좋아지면 멈춤
    )
]

## 3) SFTTrainer

In [36]:
# KeyError가 나면 위에 dataset = dataset['train'] 코드를 추가해주세요.
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,              # 학습에 사용할 데이터셋(직접 준비한 데이터 지정)
    eval_dataset = valid_dataset,               # 검증에 사용할 데이터셋(EarlyStopping 기준)
    formatting_func = formatting_prompts_func,  # 데이터를 모델 입력 형식으로 변환하는 함수
    max_seq_length= max_seq_length,
    packing = False,                            # 멀티턴 데이터 학습 시 False 권장. 여러 학습 샘플을 이어붙이지 x
    args = args,
    callbacks=callbacks,                         # earlystopping 콜백(학습 조기 종료)
    dataset_num_proc = 2                        # 전처리 cpu 코어 수
)

print("트레이너 설정 완료!")

Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/44 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/12 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
트레이너 설정 완료!


# 4. 학습

## 1) 학습 전 GPU 현황

In [37]:
# 학습 전 GPU 메모리 상태 기록 (학습 후 셀과 비교용)
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU 이름     : {gpu_stats.name}")
print(f"전체 VRAM    : {max_memory} GB")
print(f"현재 예약량  : {start_gpu_memory} GB")
print(f"남은 여유    : {round(max_memory - start_gpu_memory, 3)} GB")


GPU 이름     : Tesla T4
전체 VRAM    : 14.563 GB
현재 예약량  : 3.074 GB
남은 여유    : 11.489 GB


## 2) 학습하기

In [38]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3 | Num Epochs = 50 | Total steps = 50
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,929,536 of 1,580,643,840 (2.34% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,2.535540,2.437783
2,2.535540,2.357324
3,2.394722,2.197502
4,2.119633,2.005403
5,1.815968,1.806529
6,1.491037,1.630555
7,1.176920,1.510048
8,0.922485,1.396431
9,0.676294,1.320648
10,0.467870,1.302750


Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkpoint-1/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkpoint-2/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkpoint-3/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkpoint-4/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkpoint-5/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkpoint-6/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkpoint-7/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkpoint-8/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkpoint-9/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/ouputs/checkp

In [39]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"학습 완료!")
print(f"")
print(f"학습 시간        : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} 분")
print(f"")
print(f"전체 VRAM 사용량 : {used_memory} GB ({used_percentage} %)")
print(f"LoRA 학습 사용량 : {used_memory_for_lora} GB ({lora_percentage} %)")
print(f"   (모델 로드 제외, 순수 학습에 쓴 VRAM)")


학습 완료!

학습 시간        : 2.61 분

전체 VRAM 사용량 : 3.074 GB (21.108 %)
LoRA 학습 사용량 : 0.0 GB (0.0 %)
   (모델 로드 제외, 순수 학습에 쓴 VRAM)


## 3) 모델 저장

In [40]:
# 추론을 위한 16bit model 저장
model.save_pretrained_merged(
    "/content/model_merged_qwen_custom",
    tokenizer,
    save_method ="merged_16bit"  # merged_16bit : 추론용 전체 모델(원본 모델 + LoRA 어댑터를 합친 완성된 16bit 추론용 모델)
                                 # lora : 원본 모델은 제외하고 LoRA 어댑터 가중치만 저장(사용할 때 원본 모델이 따로 필요)
)

print("Merged model saved.")
print("Run the next cell to restart the Colab runtime, then continue from section 5.")


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/model_merged_qwen_custom/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [06:36<00:00, 396.38s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [03:26<00:00, 207.00s/it]


Unsloth: Merge process complete. Saved to `/content/model_merged_qwen_custom`
Merged model saved.
Run the next cell to restart the Colab runtime, then continue from section 5.


In [ ]:
# Colab 런타임 재시작.
# 다시 연결한 뒤에는 학습 셀을 다시 실행하지말고 5번 섹션 부터 계속 진행.
import os
import time

print("Restarting runtime. After reconnecting, continue from section 5.")
time.sleep(2)
os.kill(os.getpid(), 9)


Restarting runtime. After reconnecting, continue from section 5.


# 5. 추론

## 1) 모델 불러온 후 추론

In [1]:
# unsloth가 아직 로드되어 있으면, Transformer가 unsloth에서 패치한 forward pass를 사용하게 됨.
# 그러면 오류 발생하면서 텍스트 생성 실패할 수 있음.
import sys

if any(name == "unsloth" or name.startswith("unsloth.") for name in sys.modules):
    raise RuntimeError(
        "Unsloth is still loaded in this runtime. "
        "Restart the Colab runtime, then run only section 5 for inference."
    )

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = "/content/model_merged_qwen_custom"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    dtype = torch.bfloat16,
    device_map = "auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

print("Model loaded.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/model_merged_qwen_custom' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Model loaded.


In [2]:
input_text = tokenizer.apply_chat_template(
    [{"role":"user", "content":"해외 배송"}],
    tokenize = False     ,
    add_generation_prompt = True
)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")
print(inputs)


{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,  33883, 128792,  73669,
         127105, 151645,    198, 151644,  77091,    198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}


In [3]:
from transformers import TextStreamer

print("=== After fine-tuning ===")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 512,        # 생성할 최대 토큰 수
    use_cache = False,
    repetition_penalty = 1.3,    # 같은 표현을 반복하는 것을 줄이는 옵션(1.0 패널티 없음,  1.2~1.5 정도면 반복 억제 효과 있음)
    temperature = 0.1,           # 출력 랜덤성 조절
    do_sample = True,            # 확률 기반 샘플링 사용
)


=== After fine-tuning ===


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


물류 문제는 항상 변동성이 많아요! 주문 후 최신 상태를 확인하기 위해 아래와 같이 도움이 될 수 있어요:

1. 온라인 쇼핑몰의 공식 웹사이트나 앱에서 상품 정보 페이지로 이동하고 "배송조회" 버튼을 클릭하세요.

2. 친절한 고객 지원 팀에 문의하거나 전화번호가 있는 경우 직접 연락하실 수도 있습니다.

3. 물리적 위치에 따라 일부 국가에서는 현지 통행 관제사령부 또는 해상 경찰 등과 협력하여 운임이나 시간 등을 조정할 수 있으니 참고 부탁드립니다.

4. 특정 지역 혹은 날짜 기준으로 보낼 때에는 해당 지점별 처리 속도 차이 때문에 예상 완료일은 정확하지 않을 수 있답니다.

5. 국제 우편 사항: 가격 및 제출 가능 항목 등의 변경사항이 있을 수 있으며, 신속하게 반영될 수 있도록 적극적으로 의견 보내주세요!

6. 추가적인 요금 발생 여부 : 환율 변화, 세입자 비용 증가等因素 때문입니다.
   
7. 인공지능 서비스 이용 시에도 실제 운영시간 내외에 따른 불규칙성 가능성 존재합니다.<|im_end|>


# 2) Ollama에 내 모델 등록하기

## 1) Unsloth로 모델 불러오기

In [4]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = False

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/model_merged_qwen_custom",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

print(f"모델 로드 완료!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/model_merged_qwen_custom' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/model_merged_qwen_custom' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


모델 로드 완료!


## 2) GGUF로 모델 저장하기

In [5]:
# GGUF 변환 + 저장
model.save_pretrained_gguf(
    "/content/model_merged_qwen_custom_gguf",        # 저장 폴더 경로
    tokenizer,
    quantization_method = "q4_k_m"      # 양자화 방식(q4_k_m : 속도, 품질 균형, q8_0 : 고품질, 큰 용량)
)

Unsloth: Model is not a PEFT model. Using existing checkpoint at /content/model_merged_qwen_custom


Unsloth: Restored added_tokens_decoder metadata in /content/model_merged_qwen_custom/tokenizer_config.json.


Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9827-mix-1f1aaa4 (app-b9827-mix-1f1aaa4-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/model_merged_qwen_custom_gguf/model_merged_qwen_custom.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/model_

{'save_directory': '/content/model_merged_qwen_custom',
 'gguf_directory': '/content/model_merged_qwen_custom_gguf',
 'gguf_files': ['/content/model_merged_qwen_custom_gguf/model_merged_qwen_custom.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

In [6]:
# GGUF 변환 + 저장
model.save_pretrained_gguf(
    "/content/model_merged_qwen_teddynote_gguf_q8",
    tokenizer,
    quantization_method = "q8_0"
)


Unsloth: Model is not a PEFT model. Using existing checkpoint at /content/model_merged_qwen_custom
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/content/model_merged_qwen_custom_gguf/model_merged_qwen_custom.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/model_merged_qwen_custom_gguf/mode

{'save_directory': '/content/model_merged_qwen_custom',
 'gguf_directory': '/content/model_merged_qwen_custom_gguf',
 'gguf_files': ['/content/model_merged_qwen_custom_gguf/model_merged_qwen_custom.Q8_0.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

## 3) Ollama에 내 모델 GGUF 등록하기

In [ ]:
# 터미널에서 실행하세요
# cd models/model_merged_gguf
# ollama create 내모델이름설정 -f Modelfile
# ollama Desktop 켜보세요